# M9 — dbt setup, run, test

Run from Databricks Repos root after **Pull**. Requires bronze tables and M6 gold dimensions.

Set secrets in a cluster-scoped env or notebook widget:
- `DATABRICKS_HOST`
- `DATABRICKS_HTTP_PATH`
- `DATABRICKS_TOKEN`

In [ ]:
%pip install -q "dbt-databricks>=1.9.0,<2.0.0"

In [ ]:
import json
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path("/Workspace/Repos")  # adjust if your repo path differs
# Common pattern: find repo by walking from cwd
cwd = Path.cwd()
DBT_DIR = cwd / "dbt" if (cwd / "dbt" / "dbt_project.yml").exists() else cwd.parent / "dbt"
if not (DBT_DIR / "dbt_project.yml").exists():
    for p in [cwd, *cwd.parents]:
        if (p / "dbt" / "dbt_project.yml").exists():
            DBT_DIR = p / "dbt"
            break

PROFILES = DBT_DIR / "profiles.yml"
if not PROFILES.exists():
    example = (DBT_DIR / "profiles.yml.example").read_text()
    PROFILES.write_text(example)

def run_dbt(*args, target="dev"):
    cmd = ["dbt", *args, "--project-dir", str(DBT_DIR), "--profiles-dir", str(DBT_DIR), "--target", target]
    print(" ".join(cmd))
    out = subprocess.run(cmd, capture_output=True, text=True, check=False)
    print(out.stdout)
    if out.returncode != 0:
        print(out.stderr)
        raise RuntimeError(f"dbt failed: {' '.join(args)}")
    return out.stdout

print("dbt project:", DBT_DIR)

In [ ]:
run_dbt("debug", target="dev")
run_dbt("debug", target="prod")

In [ ]:
run_dbt("source", "freshness", target="dev")

In [ ]:
run_dbt("run", target="dev")
run_dbt("test", target="dev")

In [ ]:
run_dbt("run", "--select", "fact_sales_incremental", target="dev")
initial = spark.sql("select count(*) as n from globalmart.dbt_dev_marts.fact_sales_incremental").collect()[0].n
run_dbt("run", "--select", "fact_sales_incremental", target="dev")
rerun = spark.sql("select count(*) as n from globalmart.dbt_dev_marts.fact_sales_incremental").collect()[0].n

report = {
    "task": "dbt_incremental_fact",
    "table": "globalmart.dbt_dev_marts.fact_sales_incremental",
    "row_count_after_initial": int(initial),
    "row_count_after_rerun": int(rerun),
    "idempotent_rerun": initial == rerun,
}
print(json.dumps(report, indent=2))

In [ ]:
run_dbt("snapshot", target="dev")
snap_count = spark.sql("select count(*) as n from globalmart.dbt_dev_snapshots.snap_customers").collect()[0].n
print(json.dumps({"task": "dbt_snapshot", "snap_customers_rows": int(snap_count)}, indent=2))